# Inverse Ising (Gaussian approximation)

Companion notebook to `spatial_embedding.ipynb`. Motivation: the SVD spectrum and NMF reconstruction curve both decay smoothly with no elbow, meaning role structure here isn't concentrated in a few axes — every choice of d or k loses real information. And the role-residual ranking we introduced (`sim − λ · PPMI`) is an ad-hoc patch around the deeper issue that PMI conflates direct and indirect co-occurrence.

**The claim.** The Ising J at maximum entropy is the *direct* coupling between two Pokémon after marginalizing out all others. Its sign should structurally separate:

- **+J**: synergistic teammates (archetype cores)
- **−J**: slot competitors / role substitutes (they want to *not* be on the same team)
- **|J| ≈ 0**: structurally unrelated

Full pseudo-likelihood Ising needs per-team binary samples (Phase 2). Chaos data gives us only 2nd-order moments, so we use the leading-order **moment-matching MaxEnt** estimate: Gaussian / precision-matrix `J ≈ −Σ⁻¹` off-diagonal. Same data, same vocab cutoff as the spatial-embedding notebook, so rankings are directly comparable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

import helpers

DATA_PATH = "gen9championsvgc2026regma-1760.json"
TEAM_SIZE = 6
RNG_SEED = 0

chaos = helpers.load_chaos(DATA_PATH)
vocab = helpers.build_vocab(chaos, min_usage=0.002)
C = helpers.build_cooccurrence(chaos, vocab)
n = len(vocab)
name_to_idx = {name: i for i, name in enumerate(vocab)}

print(f"metagame: {chaos.metagame}, battles: {chaos.n_battles:,}")
print(f"vocab:    {n} pokemon")

## Binary moments

Treat each team as a binary vector over Pokémon. The Ising fit is determined by two empirical moments:

- `m[i]` = P(Pokémon *i* is on a random team), available directly from `chaos.pokemon[name]["usage"]`.
- `p_joint[i, j]` = P(both *i* and *j* are on a random team), derived from the weighted `Teammates` field.

Normalization for `p_joint`: each team contributes `team_size × (team_size − 1)` ordered (i, j) pairs to the total Teammates weight. So `weighted_teams = sum(C) / (team_size × (team_size − 1))` and `p_joint = C / weighted_teams`.

Sanity check: `sum_j p_joint[i, j]` should equal `(team_size − 1) × m[i]` (each team containing *i* contributes `team_size − 1` other mons). The chaos data is skill-weighted not raw-count, but we treat the weighting as approximately uniform across teams — if it weren't, this ratio would deviate from 1.

In [ ]:
m, p_joint = helpers.binary_moments(chaos, vocab, C, team_size=TEAM_SIZE)

row_sum = p_joint.sum(axis=1)
expected = (TEAM_SIZE - 1) * m
ratio = row_sum / expected

weighted_teams = C.sum() / (TEAM_SIZE * (TEAM_SIZE - 1))
print(f"weighted_teams = {weighted_teams:,.0f}")
print(f"row-sum / (team_size-1)*m  median = {np.median(ratio):.3f}, range = [{ratio.min():.3f}, {ratio.max():.3f}]")
print("(values near 1.0 confirm the normalization is consistent with the marginals)")

## Binary Pearson correlation matrix

For binary indicators:

$$\mathrm{Cov}(X_i, X_j) = p_\text{joint}[i,j] - m_i m_j$$
$$\mathrm{Var}(X_i) = m_i (1 - m_i)$$
$$\mathrm{Corr}[i,j] = \frac{p_\text{joint}[i,j] - m_i m_j}{\sqrt{m_i(1-m_i) \cdot m_j(1-m_j)}}$$

Inspect the eigenvalue spectrum. If `Corr` is PSD (no negative eigenvalues), we can invert it directly with minimal regularization. Otherwise we need more aggressive shrinkage.

In [ ]:
Corr = helpers.binary_correlation(m, p_joint)

eigvals = np.linalg.eigvalsh(Corr)
print(f"Corr: shape={Corr.shape}, symmetric={np.allclose(Corr, Corr.T)}")
print(f"eigenvalues: min={eigvals.min():.4f}, max={eigvals.max():.4f}")
print(f"  num negative:  {(eigvals < 0).sum()}")
print(f"  cond number:   {eigvals.max() / max(eigvals.min(), 1e-10):.2e}")
print(f"  PSD: {(eigvals >= 0).all()}")

## Regularize and invert → J

`Θ = (Corr + εI)⁻¹` is the precision matrix. For a Gaussian random field this is the (negative) coupling matrix: `J ≈ −Θ` off-diagonal. The regularization `ε` shrinks the smallest eigenvalues away from zero so the inversion is well-conditioned even when correlation has near-zero modes.

Starting with `ε = 0.01` (sensible default; sensitivity sweep deferred).

In [ ]:
eps = 0.01
J, Theta = helpers.ising_gaussian(Corr, eps=eps)

print(f"eps = {eps}")
print(f"J: shape={J.shape}, symmetric={np.allclose(J, J.T)}")
print(f"J range: [{J.min():.3f}, {J.max():.3f}]")
print(f"|J| > 0.01:  {(np.abs(J) > 0.01).sum() // 2} unordered pairs (of {n*(n-1)//2} total)")

## J vs PMI scatter — quadrant structure

Core validation. Plot `J[i, j]` against `PMI[i, j]` for all off-diagonal pairs. Predicted quadrants:

| quadrant | reading |
|---|---|
| +PMI, +J (upper right) | archetype cores — co-occur AND directly synergistic |
| +PMI, −J (lower right) | co-occurrence is fully explained by indirect paths |
| ≈0 PMI, −J (lower middle) | role substitutes — never co-occur AND structurally repulsive |
| ≈0 PMI, ≈0 J (origin) | structurally unrelated |

If J ≈ PMI everywhere (points on a single diagonal), the Gaussian approximation isn't separating direct from indirect coupling — the framing fails.

In [ ]:
P = helpers.build_ppmi(C)
iu, ju = np.triu_indices_from(J, k=1)
J_flat = J[iu, ju]
P_flat = P[iu, ju]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(P_flat, J_flat, s=4, alpha=0.25)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set(xlabel="PPMI[i, j]", ylabel="J[i, j]",
       title="Ising J vs PPMI across all pairs")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Quadrant counts
thr_p, thr_j = 0.5, 0.01
q_arch  = ((P_flat > thr_p)  & (J_flat > thr_j)).sum()
q_indir = ((P_flat > thr_p)  & (J_flat < -thr_j)).sum()
q_subst = ((P_flat <= thr_p) & (J_flat < -thr_j)).sum()
q_indep = ((P_flat <= thr_p) & (np.abs(J_flat) <= thr_j)).sum()
print(f"+PMI, +J (archetype core):       {q_arch}")
print(f"+PMI, -J (indirect-only):        {q_indir}")
print(f"~0 PMI, -J (substitute):         {q_subst}")
print(f"~0 PMI, ~0 J (independent):      {q_indep}")

## Top +J pairs — archetype synergies

Should be team-archetype cores: weather-and-abuser pairs, designated partners. PMI for these should also be high (they co-occur), distinguishing them from substitutes that look similar in cosine geometry but have direct PMI ≈ 0.

In [ ]:
order = np.argsort(-J_flat)[:25]
print(f"{'rank':>4}  {'J':>7}  {'PMI':>5}  pair")
for r, k in enumerate(order, 1):
    i, j = iu[k], ju[k]
    print(f"{r:>4}  {J[i,j]:+7.3f}  {P[i,j]:5.2f}  {vocab[i]:25s}  {vocab[j]}")

## Top −J pairs — role substitutes

Headline result. Pokémon that the model says want to *not* be on the same team — slot competitors. Should surface known substitute pairs: rain setters (Pelipper/Politoed), TR setters (Farigiraf/Oranguru), mega/base versions of the same species (mutually exclusive by game rules), and cross-archetype role-mates that pure cosine NN missed.

In [ ]:
order = np.argsort(J_flat)[:25]
print(f"{'rank':>4}  {'J':>7}  {'PMI':>5}  pair")
for r, k in enumerate(order, 1):
    i, j = iu[k], ju[k]
    print(f"{r:>4}  {J[i,j]:+7.3f}  {P[i,j]:5.2f}  {vocab[i]:25s}  {vocab[j]}")

## Per-query: synergies and substitutes

Principled replacement for the `role_score` per-query view from the prior notebook. For each query, the top +J partners are its synergistic teammates; the top −J partners are its slot competitors.

In [ ]:
queries = ["Hatterene", "Tyranitar", "Pelipper", "Sneasler", "Sinistcha", "Farigiraf"]

for q in queries:
    if q not in name_to_idx:
        print(f"({q} not in vocab)\n")
        continue
    i = name_to_idx[q]
    pos = [j for j in np.argsort(-J[i]) if j != i][:8]
    neg = [j for j in np.argsort(J[i]) if j != i][:8]
    print(f"{q}")
    print("  +J synergies:   " + ", ".join(f"{vocab[j]} ({J[i,j]:+.2f})" for j in pos))
    print("  -J substitutes: " + ", ".join(f"{vocab[j]} ({J[i,j]:+.2f})" for j in neg))
    print()

## Cross-reference: −J pairs vs role-residual ranking

If Ising J is the principled replacement for the role-residual hack, the top −J pairs and the top role-residual pairs should overlap substantially. Recompute the residual ranking from the prior notebook (`role_score = sim − λ · normalized(PPMI)` at `d=50`, `λ=1.0`) and measure Jaccard overlap with top −J.

Target: >50% Jaccard at top-25 confirms the framing recovers what the hack was approximating.

In [ ]:
d = 50
svd = TruncatedSVD(n_components=d, random_state=RNG_SEED)
US = svd.fit_transform(P)
embed = US / np.sqrt(svd.singular_values_)
norms = np.linalg.norm(embed, axis=1, keepdims=True)
embed = embed / np.where(norms > 0, norms, 1.0)
sim = embed @ embed.T

lam = 1.0
role_score = sim - lam * (P / P.max())
np.fill_diagonal(role_score, -np.inf)

top_role = set(zip(iu[np.argsort(-role_score[iu, ju])[:25]],
                   ju[np.argsort(-role_score[iu, ju])[:25]]))
top_negJ = set(zip(iu[np.argsort(J_flat)[:25]],
                   ju[np.argsort(J_flat)[:25]]))

intersection = top_role & top_negJ
union = top_role | top_negJ
jaccard = len(intersection) / len(union)

print(f"Top-25 role_score (λ={lam}, d={d})  ∩  Top-25 -J  =  {len(intersection)}")
print(f"Top-25 role_score                ∪  Top-25 -J  =  {len(union)}")
print(f"Jaccard overlap:                                   {jaccard:.2f}")

print("\nShared pairs (in both top-25 lists):")
for (i, j) in sorted(intersection, key=lambda p: J[p]):
    print(f"  J={J[i,j]:+.3f}  role={role_score[i,j]:.3f}  PMI={P[i,j]:.2f}  {vocab[i]:25s}  {vocab[j]}")

print("\n-J only (in -J top-25 but not role_score top-25):")
for (i, j) in sorted(top_negJ - top_role, key=lambda p: J[p]):
    print(f"  J={J[i,j]:+.3f}  role={role_score[i,j]:.3f}  PMI={P[i,j]:.2f}  {vocab[i]:25s}  {vocab[j]}")

print("\nrole_score only (in role_score top-25 but not -J top-25):")
for (i, j) in sorted(top_role - top_negJ, key=lambda p: -role_score[p]):
    print(f"  J={J[i,j]:+.3f}  role={role_score[i,j]:.3f}  PMI={P[i,j]:.2f}  {vocab[i]:25s}  {vocab[j]}")

## Visualizing J

Three angles on the coupling matrix:

1. **Signed distribution** — how is J distributed across pairs? Is it bimodal (concentrated synergies + concentrated substitutes) or smooth?
2. **Hierarchically-reordered heatmap** — visual community structure. Block diagonal = mons whose coupling profiles cluster together. Off-diagonal blocks indicate cross-community relationships.
3. **Sign-split network panels** — relational view. Restricted to the top 60 Pokémon by usage (the meta-relevant subset) and to the strongest 25% of each sign for legibility. Left panel: +J synergies. Right panel: −J competitions. Single-panel mixed-sign views collapse into hairballs at this size — splitting and restricting is the cleanup.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(J_flat, bins=80, log=True, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="k", lw=0.5)
axes[0].set(xlabel="J", ylabel="count (log)", title="Signed J distribution")
axes[0].grid(alpha=0.3)

axes[1].hist(np.abs(J_flat), bins=80, log=True, color="steelblue", edgecolor="white")
axes[1].set(xlabel="|J|", ylabel="count (log)", title="|J| distribution")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"+J pairs:        {(J_flat > 0).sum():,}")
print(f"-J pairs:        {(J_flat < 0).sum():,}")
print(f"|J| > 0.05:      {(np.abs(J_flat) > 0.05).sum():,}")
print(f"|J| > 0.10:      {(np.abs(J_flat) > 0.10).sum():,}")
print(f"|J| > 0.20:      {(np.abs(J_flat) > 0.20).sum():,}")

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

# Hierarchical clustering on cosine similarity of J rows (each row = a mon's coupling profile)
J_norm = J / np.linalg.norm(J, axis=1, keepdims=True).clip(min=1e-10)
row_sim = J_norm @ J_norm.T
dist = np.clip(1 - row_sim, 0, None)
np.fill_diagonal(dist, 0)
dist = (dist + dist.T) / 2

condensed = dist[np.triu_indices_from(dist, k=1)]
Z = linkage(condensed, method="average")
order = leaves_list(Z)

J_reordered = J[order][:, order]

fig, ax = plt.subplots(figsize=(10, 10))
vmax = np.abs(J).max()
im = ax.imshow(J_reordered, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="equal")
plt.colorbar(im, ax=ax, shrink=0.7, label="J")
ax.set_title(f"Ising J matrix, hierarchically reordered ({n}×{n})")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

TOP_N = 60        # restrict to top-N Pokémon by usage (meta-relevant subset)
POS_QUANTILE = 0.75   # keep top (1 - POS_QUANTILE) fraction of +J values
NEG_QUANTILE = 0.95   # keep top (1 - NEG_QUANTILE) fraction of -J magnitudes
                      # -- different per sign because the -J distribution is much denser
                      # than +J (team-budget pressure creates weak competition between
                      # most pairs); a tighter threshold balances visual density.

top_idx = np.argsort(-m)[:TOP_N]
sub_vocab = [vocab[i] for i in top_idx]
sub_m = m[top_idx]
J_sub = J[np.ix_(top_idx, top_idx)]

iu_s, ju_s = np.triu_indices(TOP_N, k=1)
J_flat_sub = J_sub[iu_s, ju_s]
pos_vals = J_flat_sub[J_flat_sub > 0]
neg_vals = -J_flat_sub[J_flat_sub < 0]
thr_pos = float(np.quantile(pos_vals, POS_QUANTILE)) if pos_vals.size else 0.0
thr_neg = float(np.quantile(neg_vals, NEG_QUANTILE)) if neg_vals.size else 0.0


def build_subgraph(keep_fn) -> nx.Graph:
    G = nx.Graph()
    for i in range(TOP_N):
        G.add_node(i, usage=float(sub_m[i]))
    for k in range(len(iu_s)):
        a, b = int(iu_s[k]), int(ju_s[k])
        v = float(J_sub[a, b])
        if keep_fn(v):
            G.add_edge(a, b, weight=abs(v))
    G.remove_nodes_from([i for i, d in G.degree() if d == 0])
    return G


G_pos = build_subgraph(lambda v: v >= thr_pos)
G_neg = build_subgraph(lambda v: -v >= thr_neg)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
panels = [
    (axes[0], G_pos, "#1f77b4", "solid",
     f"+J synergies  ({G_pos.number_of_edges()} edges, J ≥ {thr_pos:.3f})"),
    (axes[1], G_neg, "#d62728", "dashed",
     f"-J competitions  ({G_neg.number_of_edges()} edges, -J ≥ {thr_neg:.3f})"),
]

for ax, G, color, style, title in panels:
    if G.number_of_edges() == 0:
        ax.text(0.5, 0.5, "no edges above threshold", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title); ax.axis("off")
        continue
    layout = nx.kamada_kawai_layout(G, weight="weight")
    widths = [3 * d["weight"] for _, _, d in G.edges(data=True)]
    sizes = [80 + 4000 * G.nodes[i]["usage"] for i in G.nodes()]
    nx.draw_networkx_edges(G, layout, edge_color=color, alpha=0.55, width=widths, style=style, ax=ax)
    nx.draw_networkx_nodes(G, layout, node_size=sizes, node_color="#333", alpha=0.75, ax=ax)
    nx.draw_networkx_labels(G, layout, labels={i: sub_vocab[i] for i in G.nodes()}, font_size=8, ax=ax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print(f"+J subgraph: {G_pos.number_of_nodes()} nodes, {G_pos.number_of_edges()} edges, avg degree {2*G_pos.number_of_edges()/max(G_pos.number_of_nodes(), 1):.1f}")
print(f"-J subgraph: {G_neg.number_of_nodes()} nodes, {G_neg.number_of_edges()} edges, avg degree {2*G_neg.number_of_edges()/max(G_neg.number_of_nodes(), 1):.1f}")

## Per-query ego networks

For per-Pokémon exploration. Plots the Ising J neighborhood of a single mon: query at center, top `k_pos` +J partners in a blue upper arc, top `k_neg` −J partners in a red dashed lower arc. Edge widths scale with |J|, labels show the J value. Skirts the global-density problem entirely — only ever ~17 nodes on screen — and is the natural visual companion to the per-query text output above.

`plot_j_ego("PokemonName")` to call interactively; below the function definition we call it on a few example queries.

In [ ]:
def plot_j_ego(query: str, k_pos: int = 8, k_neg: int = 8) -> None:
    """Plot the Ising J neighborhood of one Pokemon: top k_pos +J partners (upper
    arc, blue solid) and top k_neg -J partners (lower arc, red dashed)."""
    if query not in name_to_idx:
        print(f"({query} not in vocab)")
        return
    qi = name_to_idx[query]
    pos_partners = [int(j) for j in np.argsort(-J[qi]) if j != qi][:k_pos]
    neg_partners = [int(j) for j in np.argsort(J[qi]) if j != qi][:k_neg]

    layout: dict[int, tuple[float, float]] = {qi: (0.0, 0.0)}
    for idx, j in enumerate(pos_partners):
        angle = np.deg2rad(30 + 120 * idx / max(k_pos - 1, 1))
        layout[j] = (1.3 * np.cos(angle), 1.3 * np.sin(angle))
    for idx, j in enumerate(neg_partners):
        angle = np.deg2rad(210 + 120 * idx / max(k_neg - 1, 1))
        layout[j] = (1.3 * np.cos(angle), 1.3 * np.sin(angle))

    max_abs = max(abs(J[qi]).max(), 1e-10)
    fig, ax = plt.subplots(figsize=(12, 9))

    for j in pos_partners:
        w = 1 + 6 * abs(J[qi, j]) / max_abs
        ax.plot([0, layout[j][0]], [0, layout[j][1]],
                color="#1f77b4", alpha=0.65, lw=w, zorder=1)
    for j in neg_partners:
        w = 1 + 6 * abs(J[qi, j]) / max_abs
        ax.plot([0, layout[j][0]], [0, layout[j][1]],
                color="#d62728", alpha=0.65, lw=w, linestyle="dashed", zorder=1)

    for i, (x, y) in layout.items():
        if i == qi:
            color, size = "gold", 1200
        elif i in pos_partners:
            color, size = "#aec7e8", 500
        else:
            color, size = "#ff9896", 500
        ax.scatter(x, y, s=size, c=color, edgecolors="black", linewidths=0.8, zorder=10)
        label = vocab[i] if i == qi else f"{vocab[i]}\n{J[qi, i]:+.2f}"
        ax.annotate(label, (x, y), ha="center", va="center",
                    fontsize=9, zorder=11, fontweight="bold" if i == qi else "normal")

    ax.set_title(f"{query}: top {k_pos} +J synergies (top arc, blue) and top {k_neg} -J substitutes (bottom arc, red dashed)")
    ax.set_xlim(-1.7, 1.7)
    ax.set_ylim(-1.7, 1.7)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


for q in ["Froslass-Mega", "Sinistcha", "Hydreigon", "Tyranitar"]:
    plot_j_ego(q)

## Diagnostics: field h and regularization sensitivity

Two checks now that the J recovery looks structurally clean:

- **Field h**, derived via mean-field self-consistency `h_i ≈ logit(m_i) − Σⱼ Jᵢⱼ mⱼ`. Most variation should track marginal usage; deviations highlight mons whose presence is more (or less) than their interactions alone predict.
- **Regularization sweep** — confirm the top-25 ±J rankings are stable across reasonable choices of `eps`. A stable middle range is the trustable operating point.

In [ ]:
logit_m = np.log(m / (1 - m))
h = logit_m - J @ m
correction = J @ m

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(np.log10(m), h, s=28, alpha=0.7)
slope, intercept = np.polyfit(np.log10(m), h, 1)
residual = h - (intercept + slope * np.log10(m))
top_dev = np.argsort(-np.abs(residual))[:8]
for i in top_dev:
    axes[0].annotate(vocab[i], (np.log10(m[i]), h[i]), fontsize=8, alpha=0.8,
                     xytext=(4, 4), textcoords="offset points")
axes[0].set(xlabel="log10(usage)", ylabel="field h", title="Field h vs log usage")
axes[0].grid(alpha=0.3)

axes[1].scatter(logit_m, h, s=28, alpha=0.7)
xs = np.array([logit_m.min(), logit_m.max()])
axes[1].plot(xs, xs, "k--", lw=0.5, label="h = logit(m)  [no interactions]")
axes[1].set(xlabel="logit(m)", ylabel="field h",
            title="Interaction correction = logit(m) − h = J·m")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"interaction correction (J @ m): range=[{correction.min():.3f}, {correction.max():.3f}], "
      f"|mean|={np.abs(correction).mean():.3f}")
print(f"top |correction|:")
for i in np.argsort(-np.abs(correction))[:8]:
    print(f"  {correction[i]:+.3f}  {vocab[i]}  (usage={m[i]*100:.2f}%)")

In [ ]:
eps_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]

J_by_eps = {e: helpers.ising_gaussian(Corr, eps=e)[0] for e in eps_values}

base_eps = 0.01
base_neg = set(zip(iu[np.argsort(J_by_eps[base_eps][iu, ju])[:25]],
                   ju[np.argsort(J_by_eps[base_eps][iu, ju])[:25]]))
base_pos = set(zip(iu[np.argsort(-J_by_eps[base_eps][iu, ju])[:25]],
                   ju[np.argsort(-J_by_eps[base_eps][iu, ju])[:25]]))

print(f"Top-25 stability vs baseline eps={base_eps}:")
print(f"{'eps':>7}  {'-J overlap':>14}  {'+J overlap':>14}  {'J range':>22}")
for e in eps_values:
    Je = J_by_eps[e]
    neg_e = set(zip(iu[np.argsort(Je[iu, ju])[:25]], ju[np.argsort(Je[iu, ju])[:25]]))
    pos_e = set(zip(iu[np.argsort(-Je[iu, ju])[:25]], ju[np.argsort(-Je[iu, ju])[:25]]))
    print(f"{e:>7}  {len(neg_e & base_neg)/25:>5.2f} ({len(neg_e & base_neg):>2}/25)  "
          f"{len(pos_e & base_pos)/25:>5.2f} ({len(pos_e & base_pos):>2}/25)  "
          f"[{Je.min():+6.3f}, {Je.max():+6.3f}]")